In [1]:
import requests
import json
import time
import urllib3
import threading
from threading import Lock

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
WAZUH_IP = "https://10.4.47.133:55000"

USERNAME = "ai_user"
PASSWORD = "Fyp@112233"

FETCH_INTERVAL = 5
BATCH_INTERVAL = 60

RUNNING = True

In [3]:
log_buffer = []
buffer_lock = Lock()

seen_logs = set()

In [4]:
def get_token():

    r = requests.post(
        f"{WAZUH_IP}/security/user/authenticate",
        auth=(USERNAME, PASSWORD),
        verify=False
    )

    response = r.json()

    return response["data"]["token"]

In [5]:
def get_logs(token):

    headers = {
        "Authorization": f"Bearer {token}"
    }

    r = requests.get(
        f"{WAZUH_IP}/manager/logs?limit=50",
        headers=headers,
        verify=False
    )

    data = r.json()

    return data.get("data", {}).get("affected_items", [])

In [6]:
def fusion(network, cloud, host):

    score = 0

    if network != "Benign":
        score += 60

    if cloud == "Suspicious":
        score += 20

    if host == -1:
        score += 20

    if score >= 70:
        ai_alert = "HIGH"

    elif score >= 40:
        ai_alert = "MEDIUM"

    else:
        ai_alert = "LOW"

    if network != "Benign":
        source_layer = "network"

    elif host == -1:
        source_layer = "host"

    else:
        source_layer = "cloud"

    return score, ai_alert, source_layer

In [ ]:
def process_logs(logs):

    enriched_logs = []

    for log in logs:

        msg = str(log.get("description", "")).lower()

        print("\n🔍 RAW LOG:", msg)

        # IMPORTANT FILTER
        if not any(keyword in msg for keyword in [           
            "authentication failed",
            "failed password",
            "invalid user",
            "unauthorized",
            "ssh",
            "attack",
            "ddos",
            "flood"
        ]):

            print("⏭️ Skipped")
            continue
        
        print("✅ PASSED FILTER")
        


        # CLOUD ANALYSIS
        if "error" in msg or "warning" in msg:
            cloud = "Suspicious"
        else:
            cloud = "Normal"

        # HOST ANALYSIS
        if any(x in msg for x in [
            "authentication",
            "failed",
            "password",
            "ssh",
            "unauthorized"
        ]):
            host = -1
        else:
            host = 1

        # NETWORK ANALYSIS
        if any(x in msg for x in [
            "ddos",
            "flood",
            "attack"
        ]):
            network = "DDoS"
        else:
            network = "Benign"

        # FUSION
        score, ai_alert, source_layer = fusion(
            network,
            cloud,
            host
        )

        event = {

            "source": "ai_fusion",

            "message": msg,

            "ai_alert": ai_alert,

            "risk_score": score,

            "source_layer": source_layer,

            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        enriched_logs.append(event)

    return enriched_logs

In [8]:
def write_to_file(events):

    path = "ai_fusion.log"

    with open(path, "a") as f:

        for event in events:

            f.write(json.dumps(event) + "\n")

    print(f"📁 Written {len(events)} logs")

In [9]:
def fetch_logs(token):

    global log_buffer
    global seen_logs

    while RUNNING:

        try:

            logs = get_logs(token)

            new_logs = []

            for log in logs:

                unique_id = (
                    str(log.get("timestamp")) +
                    str(log.get("description"))
                )

                if unique_id not in seen_logs:

                    seen_logs.add(unique_id)

                    new_logs.append(log)

            with buffer_lock:

                log_buffer.extend(new_logs)

            print(
                f"📥 Buffered: {len(log_buffer)} "
                f"(new: {len(new_logs)})"
            )

        except Exception as e:

            print("❌ Fetch Error:", e)

        time.sleep(FETCH_INTERVAL)

In [10]:
def process_batches():

    global log_buffer

    while RUNNING:

        time.sleep(BATCH_INTERVAL)

        with buffer_lock:

            snapshot = log_buffer.copy()

            log_buffer.clear()

        if not snapshot:

            print("⚠️ No logs to process")

            continue

        print(f"\n🧠 Processing {len(snapshot)} logs...")

        enriched = process_logs(snapshot)

        if enriched:

            write_to_file(enriched)

        else:

            print("⚠️ No important logs found")

In [ ]:
token = get_token()

print("✅ System started...")

fetch_thread = threading.Thread(
    target=fetch_logs,
    args=(token,)
)

process_thread = threading.Thread(
    target=process_batches
)

fetch_thread.start()

process_thread.start()

✅ System started...


📥 Buffered: 48 (new: 48)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)
📥 Buffered: 48 (new: 0)

🧠 Processing 48 logs...

🔍 RAW LOG:  ending rootcheck scan.
⏭️ Skipped

🔍 RAW LOG:  indexerconnector initialized successfully for index: wazuh-states-inventory-hotfixes-server-vmware-virtual-platform.
⏭️ Skipped

🔍 RAW LOG:  indexerconnector initialized successfully for index: wazuh-states-inventory-hardware-server-vmware-virtual-platform.
⏭️ Skipped

🔍 RAW LOG:  indexerconnector initialized successfully for index: wazuh-states-inventory-protocols-server-vmware-virtual-platform.
⏭️ Skipped

🔍 RAW LOG:  indexerconnector initialized successfully for index: wazuh-states-inventory-interfaces-server-vmware-virtual-platform.
⏭️ Skipped

🔍 RAW LOG:  indexerconnector initialized successfully for index: wazuh-

In [17]:
def stop_system():

    global RUNNING

    RUNNING = False

    print("🛑 Stopping system...")

    fetch_thread.join(timeout=2)

    process_thread.join(timeout=2)

    print("✅ System fully stopped")

stop_system()

🛑 Stopping system...
✅ System fully stopped
